# Semantic Similarity with `gemini-embedding-001`

## Setup

In [15]:
from google import genai
from google.genai import types
from pprint import pprint
from pydantic import BaseModel
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [2]:
client = genai.Client()

## A Basic Test

More or less adapted from the example code on Google's documentation.

In [4]:
example_text1 = "I have a cute cat named Mouchan."
example_text2 = "My pet kitty Mouchan is so cute."
example_texts = [example_text1, example_text2]

In [19]:
def get_embeddings(texts: list[str]) -> types.EmbedContentResponse:
    return client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts,
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
        )

In [36]:
result = get_embeddings(texts=example_texts)

In [49]:
df = pd.DataFrame(
    cosine_similarity([e.values for e in result.embeddings]),
    index=example_texts,
    columns=example_texts,
)

In [50]:
df

,I have a cute cat named Mouchan.,My pet kitty Mouchan is so cute.
I have a cute cat named Mouchan.,1.000000,0.958796
My pet kitty Mouchan is so cute.,0.958796,1.000000


## A More Sophsticated Test

Start with a series of texts, get their embeddings, and then check if a new text overlaps (has a high similarity score) to any of them.

In [5]:
example_texts2 = example_texts + [
    "I have a cute dog.",
    "I love my cat.",
    "You have to take your cat to the vet.",
    "Let's get tacos."
]

Create a data class (with pydantic) to hold both text and embeddings.

In [6]:
class TextWithEmbeddings(BaseModel):
    text: str
    embeddings: types.EmbedContentResponse

Create a function that takes in a list of texts and returns a list of the container data class defined above.

In [21]:
def get_embeddings2(texts: list[str]) -> list[TextWithEmbeddings]:
    results: list[TextWithEmbeddings] = []

    for text in texts:
        embedding: types.EmbedContentResponse = client.models.embed_content(
            model="gemini-embedding-001",
            contents=[text],
            config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
            )
        results.append(TextWithEmbeddings(text=text, embeddings=embedding))
    return results

Define a helper function to extact the embedding values themselves, a list of floats. This data turned out to be fairly deeply nested. Consider a different abstraction in future projects, or perhaps instead of a data class, use an entity with a method that returns the list of floats. (That may be the cleanest, now that I think about it).

In [24]:
def get_embedding_value(text_with_emb: TextWithEmbeddings) -> list[float]:
    data = text_with_emb.embeddings.embeddings[0].values
    if data == None:
        raise ValueError("No data in input TextWithEmbeddings object.")
    return data

Finally, define a function to check the similarity between a list of text and a new text to check, and display the results as a pandas DataFrame.

In [25]:
def get_similarity2(list_of_texts_to_check: list[str], new_text_to_check: str):
    list_of_embeddings = get_embeddings2(texts=list_of_texts_to_check)
    new_embedding = get_embeddings2(texts=[new_text_to_check])

    new_embedding_value = get_embedding_value(text_with_emb=new_embedding[0])

    similarity_results = []

    for embedding in list_of_embeddings:
        embed_value = get_embedding_value(text_with_emb=embedding)
        similarity = cosine_similarity([embed_value, new_embedding_value])
        data = {
            "Text 1": embedding.text,
            "Text 2": new_embedding[0].text,
            "Similarity": similarity,
        }
        similarity_results.append(data)

    return similarity_results

In [26]:
new_text = "This is my cat."
similarity_results = get_similarity2(list_of_texts_to_check=example_texts2, new_text_to_check=new_text)

In [27]:
pd.DataFrame({
    "Text": [r["Text 1"] for r in similarity_results],
    "Similarity": [r["Similarity"][0][1] for r in similarity_results],
}).sort_values("Similarity", ascending=False)

,Text,Similarity
3,I love my cat.,0.922544
0,I have a cute cat named Mouchan.,0.908285
1,My pet kitty Mouchan is so cute.,0.876308
2,I have a cute dog.,0.848890
4,You have to take your cat to the vet.,0.847107
5,Let's get tacos.,0.726863


Some surprising results here. "Let's get tacos." gets a 0.73 similarity score, much higher than expected.